# Searching for the Higgs boson in the H&#8594;&gamma;&gamma; channel
<br>
<CENTER>
    <a href="http://opendata.atlas.cern/release/2020/documentation/notebooks/intro.html" class="icons"><img src="https://github.com/atlas-outreach-data-tools/notebooks-collection-opendata/blob/master/images/ATLASOD.gif?raw=1" style="width:40%"></a>
</CENTER>

The Higgs boson can be created several different ways. We represent these different production mechanisms as Feynman diagrams that show particles interacting. Here are the four most important for the Standard Model Higgs boson:<br><br>

<CENTER><img src="https://github.com/atlas-outreach-data-tools/notebooks-collection-opendata/blob/master/images/Figures_FeynmanHprod.png?raw=1" style="width:50%"></CENTER>

Clockwise from the top right, those show two gluons creating a Higgs boson via a loop (the triangle; this diagram is also known as gluon-gluon fusion), the Higgs boson being radiated from a vector boson like a W or Z boson (this production mechanism comes with two forward quarks, which appear in the detector as "jets", and is known as vector-boson fusion), the Higgs boson being radiated from a top quark (the process is also known as "ttH" because of the two top quarks in the event), and the Higgs boson being radiated from a vector boson without the two additional quarks (in this case, the process is known as "associated production", because the Higgs boson is being made in association with a vector boson).

In this notebook, we will look for the Higgs boson decaying to two photons. The analysis is relatively straightforward: we look for exactly two photons in an event and ask that they are isolated. We then look at the invariant mass of the two photons to see if they might have originated from a Higgs boson. This analysis is similar to the [uproot-based Higgs$\rightarrow\gamma\gamma$ analysis](https://opendata.atlas.cern/docs/13TeV25Doc/StandardModel#higgs-to-%CE%B3%CE%B3-analysis-new).

## Implementation

We'll start by setting up the packages we need to work with the open data. In this case, we'll install `ROOT` (a standard analysis package used in particle physics) and `atlasopenmagic` (a package for working with the ATLAS open data). The setup is a big complicated so that it will work on either Binder or Colab:

* SWAN and Binder provide environments based on containers that include some of the software we want, while in Colab we need to install it.
* On Binder and SWAN we can use `%jsroot` magic to allow us to interact with the histograms that we draw (e.g. zoom).
* Binder doesn't allow us to use xrootd to access files, so we need to access them via the https protocol. Using the ROOT protocol will allow us to read files that are remote, rather than having to copy them locally before opening them.
* On SWAN, we have to do a bit of extra work when pip-installing packages

You don't need to worry about the details above if you don't understand them, but those are the adjustments we'll make in the next cell.

We'll also start a timer, so that we can report how long it takes the notebook to run.

In [ ]:
# Our environment will depend on whether we're running in Colab, Binder, or SWAN
if 'google.colab' in str(get_ipython()):
    print('Running on Colab - will install ROOT')
    %pip install ROOT
    # Colab allows us to use the (easier) ROOT protocol to read files
    protocol = 'ROOT'
    import ROOT
    %jsroot off
else:
    print('Not running on Colab')
    # On Binder we have to use the https protocol to read files
    protocol = 'https'
    import ROOT
    # But on Binder we can enable jsroot, which lets us interact with the
    # histograms we draw later on
    %jsroot on

# If we are on SWAN, then we need a little extra work to install atlasopenmagic
import os
if 'SWAN_ENV_FILE' in os.environ:
    %pip install --user atlasopenmagic
    import sys
    sys.path += [ f'{os.environ["HOME"]}/.local/lib/python{sys.version_info.major}.{sys.version_info.minor}/site-packages' ]
else:
    %pip install atlasopenmagic
# We'll also use the time module to report how long this notebook takes to run
import time
start = time.time()

Next, we'll use `atlasopenmagic` to get a file from the Open Data. If you'd like to understand what these lines do a bit better, you can go through the [metadata tutorial](https://opendata.atlas.cern/docs/13TeV25Doc/Concepts#accessing-metadata-new). Here we will also use the `GamGam` skim, which gives us files that contain events that all have at least two photons. After we have the file address, we'll open it using ROOT.

In [ ]:
# Use atlasopenmagic to get a file for us to look at
import atlasopenmagic as atom
atom.set_release('2025e-13tev-beta')
file_names = atom.get_urls('data',protocol=protocol,skim='GamGam',cache=False)

In the files, our data are kept in a tree called `analysis`. We will use a "chain" to loop over all the files and get the information from that tree. We can also ask the chain how main entries (events) it has. This can take a bit of time, as all the files have to be checked to calculate the total number of events, so it is not done by default.

In [ ]:
# Create a chain containing the analysis trees
chain = ROOT.TChain("analysis")
# Add all our input files to the chain
for file_name in file_names:
    chain.Add(file_name)
# If you wish, you can check how many events the chain contains
#chain.GetEntries()

Now we'll create a blank canvas on which we can draw the histogram, and we'll make an extra histogram that we can fill up with our data.

In [ ]:
# Blank canvas
canvas = ROOT.TCanvas("Canvas","cz",800,600)
#Invariant mass histograms definition
hist = ROOT.TH1F("h_M_Hyy","Diphoton invariant-mass ; Invariant Mass m_{yy} [GeV] ; Events",55,105,160)

Next, we'll loop over the events in our file (in the tree) and look for events that have exactly two photons. We'll check that those two photons are also isolated (they do not have lots of other particles around them), which is a good indication that they might have come from a Higgs boson decay. Once we have selected our two photons, we'll calculate their invariant mass, which we can plot in our histogram. If everything has gone well, we should see a small peak around the mass of the Higgs boson in our histogram.

In case you would like to see what variables are available in the Open Data files, you can read the [documentation here](https://opendata.atlas.cern/docs/data/for_education/13TeV25_details).

In [ ]:
# To save memory and go a little bit faster, we will make two Lorentz vectors
# here and re-use them during the loop over events
Photon_1 = ROOT.TLorentzVector()
Photon_2 = ROOT.TLorentzVector()

# To make this run a bit faster, we will disable any branches we don't need
chain.SetBranchStatus('*',0)
# Now let's enable exactly the branches we want
for variable in ['photon_pt', 'photon_eta', 'photon_phi',
                 'photon_isTightID', 'photon_ptcone20']:
    chain.SetBranchStatus(variable,1)

# In case you'd like to stop a bit earlier, before going through all the data,
# set this variable to a positive number
stop_early = 100000

# Now we'll loop through all the events, keeping the event number in mind
for n,event in enumerate(chain):
    # Check if we should stop
    if n>stop_early and stop_early>0:
        break
    # Print the number of events we have processed so far
    if n%10000==0:
        print(f'Processing event {n}')

    # Check if our highest momentum photon passes basic momentum, isolation, and
    # location (pseudo-rapidity, or eta) requirements
    if chain.photon_isTightID[0]\
      and chain.photon_pt[0] > 50. and abs(chain.photon_eta[0]) < 2.37\
      and (abs(chain.photon_eta[0]) < 1.37 or abs(chain.photon_eta[0]) > 1.52)\
      and chain.photon_ptcone20[0]/chain.photon_pt[0]<0.055:
        # Good first photon! Let's see if our second photon also passes a
        # similar set of requirements We had a good photon!
        if chain.photon_isTightID[1]\
          and chain.photon_pt[1] > 30. and abs(chain.photon_eta[1]) < 2.37\
          and (abs(chain.photon_eta[1]) < 1.37 or abs(chain.photon_eta[1]) > 1.52)\
          and chain.photon_ptcone20[1]/chain.photon_pt[1]<0.055:
            # Good second phtoon!
            # Set the Lorentz vectors based on our photon kinematics.
            # We can take advantage of the fact that photons always have
            # zero mass to make the vector a bit simpler.
            Photon_1.SetPtEtaPhiM(chain.photon_pt[0],
                                  chain.photon_eta[0],
                                  chain.photon_phi[0],
                                  0)
            Photon_2.SetPtEtaPhiM(chain.photon_pt[1],
                                  chain.photon_eta[1],
                                  chain.photon_phi[1],
                                  0)
            # Add the two TLorentz vectors
            Photon_12 = Photon_1 + Photon_2
            # Finally, we'll require that the individual photon momentum is
            # at least 35% of the invariant mass. This requires some
            # symmetry, which is more common in Higgs boson decays than in
            # other Standard Model events.
            if chain.photon_pt[0]/Photon_12.M() > 0.35\
              and chain.photon_pt[1]/Photon_12.M() > 0.35:
                # Fill the histogram with the mass of the gamma-gamma system
                hist.Fill(Photon_12.M())
        # End of the isolation requirement
    # End of the check for 2-good photons
# End of the loop over the events

### Fitting and plotting

Before we draw our histogram, we'll perform a fit to the data. The fit uses a Gaussian distribution for the Higgs boson "signal" (what we're looking for) and a 4th order polynomial for the other Standard Model backgrounds. There is quite a bit of work that goes into the analysis before it's published to ensure that the 4th order polynominal for the background is the right function: it has to have enough flexibility to capture the shape of the background, but not so much that it accidentally hides or artificially inflates the Higgs boson signal.

The ROOT libraries have quite a bit of helpful functionality for fitting functions to histograms, so we can make use of that. Don't worry if you don't understand all of it. You might see afterwards if you're able to make simple modifications to these fits (for example, trying a 3rd order polynominal) to see what happens.

In [ ]:
# Convert histogram into RooDataHist
x = ROOT.RooRealVar("m_yy", "di-photon invariant mass m_{#gamma#gamma} [GeV]", 100, 160)
data = ROOT.RooDataHist("data", "data", ROOT.RooArgList(x), hist)

# --- Define signal model (Gaussian) ---
mean   = ROOT.RooRealVar("mean", "mean of gauss", 125, 120, 130)
width  = ROOT.RooRealVar("width", "width of gauss", 2.0, 0.5, 5.0)
gauss  = ROOT.RooGaussian("gauss", "gaussian PDF", x, mean, width)

# --- Define background model (4th order polynomial) ---
a0 = ROOT.RooRealVar("a0", "a0", 0.0, -1, 1)
a1 = ROOT.RooRealVar("a1", "a1", 0.0, -1, 1)
a2 = ROOT.RooRealVar("a2", "a2", 0.0, -1, 1)
a3 = ROOT.RooRealVar("a3", "a3", 0.0, -1, 1)
a4 = ROOT.RooRealVar("a4", "a4", 0.0, -1, 1)
bkg = ROOT.RooChebychev("bkg", "Chebychev background", x, ROOT.RooArgList(a0,a1,a2,a3,a4))

# --- Define coefficients (yields) ---
nsig = ROOT.RooRealVar("nsig", "signal yield", hist.Integral()/20., 0., hist.Integral())
nbkg = ROOT.RooRealVar("nbkg", "background yield", hist.Integral(), 0., hist.Integral())

# --- Total model ---
model = ROOT.RooAddPdf("model", "sig + bkg", ROOT.RooArgList(gauss, bkg), ROOT.RooArgList(nsig, nbkg))

# --- Perform the fit ---
fit_result = model.fitTo(data, ROOT.RooFit.Extended(True), ROOT.RooFit.Save(), ROOT.RooFit.PrintLevel(-1))

Now that the fit is done, we can ask ROOT to tell us what values the functions took. The coefficients of the background polynominal aren't terribly interesting on their own, but they're important to properly describe the background. The mean of the Gaussian distribution tells us what the fitted Higgs boson mass is. The width of the Gaussian distribution tells us how "wide" the signal is — how spread out it is.

Before you run this cell, you might take a moment to think about what you expect, in particular for the width. Particles have a natural "width", which is related to how they decay. You can look up the width of the Higgs boson. Why might the width that we observe in this fit be significantly different from the width in the Standard Model?

In [ ]:
print("mean: ", mean.getVal())
print("width: ", width.getVal())
print("polynomic function (using the Chebyshev polynomials basis): ")
print(f"{a0.getVal()} + {a1.getVal()}*T_1(x) + {a2.getVal()}*T_2(x) + {a3.getVal()}*T_3(x) + {a4.getVal()}*T_4(x)")

All we have to do now is draw our histogram on the canvas that we made, and then draw (display) the canvas. ROOT has lots of drawing options for histograms; you can read about them [here](https://root.cern.ch/doc/master/classTHistPainter.html). Most of the work in this cell goes into making the histogram look pretty — getting things shifted around on the canvas nicely, making sure things have the right colors, and drawing the nice background-subtracted signal that you see in the bottom panel of the figure.

In [ ]:
# =======================
# Main Pad (Data + Fit)
# =======================
canvas = ROOT.TCanvas("canvas", "canvas", 800, 600)
    
mainPad = ROOT.TPad("mainPad", "mainPad", 0, 0.3, 1, 1.0)
mainPad.SetBottomMargin(0)
mainPad.Draw()
mainPad.cd()

# Get histogram representation of data and background
# Ensure both histograms use the same binning

nbins = hist.GetNbinsX()
xmin = hist.GetXaxis().GetXmin()
xmax = hist.GetXaxis().GetXmax()

h_data = data.createHistogram("h_data", x, ROOT.RooFit.Binning(nbins, xmin, xmax));
h_bkg  = bkg.createHistogram("h_bkg", x, ROOT.RooFit.Binning(nbins, xmin, xmax));

print(f"nbins h_data: {h_data.GetNbinsX()}")
print(f"nbins h_bkg: {h_bkg.GetNbinsX()}")

print("===================================================================")
print(f"integral h_bkg: {h_bkg.Integral()}")
print(f"nbkg expected: {nbkg}")

# Normalize background histogram to expected number of events. The h_bkg is normalized to 1, so 
# that's why we need to scale it with the nbkg estimation given in the fitted model
h_bkg.Scale(nbkg.getVal() / h_bkg.Integral())

# Create residual histogram (data - bkg)
resid = ROOT.TH1F("residual","Diphoton invariant-mass ; Invariant Mass m_{yy} [GeV] ; events",60,100,160)

for i in range(h_data.GetNbinsX()):
    data_value = h_data.GetBinContent(i)
    bkg_value = h_bkg.GetBinContent(i)
    
    # residual value
    res_value  = data_value - bkg_value
    resid.SetBinContent(i, res_value)

# --- Plot results ---
frame = x.frame()
data.plotOn(frame, ROOT.RooFit.MarkerStyle(20), ROOT.RooFit.MarkerSize(0.5))
model.plotOn(frame, ROOT.RooFit.Name("sig_bkg_curve"), ROOT.RooFit.LineWidth(2), 
             ROOT.RooFit.LineColor(ROOT.kBlue))
model.plotOn(frame, ROOT.RooFit.Components("bkg"), ROOT.RooFit.LineWidth(2), ROOT.RooFit.LineStyle(ROOT.kDashed), 
             ROOT.RooFit.LineColor(ROOT.kRed), ROOT.RooFit.Name("bkg_curve"))
model.plotOn(frame, ROOT.RooFit.Components("gauss"), ROOT.RooFit.LineStyle(ROOT.kDotted), 
             ROOT.RooFit.LineColor(ROOT.kBlue), ROOT.RooFit.Name("gaussian_curve"))

frame.SetTitle("Di-photon invariant mass m_{#gamma#gamma}")
frame.GetYaxis().SetTitle("Events / bin")
frame.GetXaxis().SetTitleSize(0.05)   # size of x-axis title
frame.GetYaxis().SetTitleSize(0.05)   # size of y-axis title
frame.GetXaxis().SetLabelSize(0.03)   # size of x-axis labels
frame.GetYaxis().SetLabelSize(0.03)   # size of y-axis labels
frame.GetXaxis().SetTitleFont(42)     # Helvetica (ROOT default)
frame.GetYaxis().SetTitleFont(42)

frame.Draw()
canvas.cd()

# Legend
sig_bkg_curve = frame.findObject("sig_bkg_curve")
bkg_curve = frame.findObject("bkg_curve")

legend = ROOT.TLegend(0.55, 0.75, 0.85, 0.90)
#legend.AddEntry(frame.findObject("h_data"), "Data", "lep")
legend.AddEntry(frame, "Data", "lep")
legend.AddEntry(sig_bkg_curve, "Sig+Bkg Fit (m_{H}=125 GeV)", "l")
legend.AddEntry(bkg_curve, "Bkg (4th order polynomial)", "l")
legend.Draw()

# Labels
latex = ROOT.TLatex()
latex.SetTextSize(0.03)
latex.DrawLatexNDC(0.60, 0.70, "ATLAS Open Data")
latex.DrawLatexNDC(0.60, 0.65, "#sqrt{s}=13 TeV, #int L dt = 36 fb^{-1}")
latex.DrawLatexNDC(0.60, 0.60, "H #rightarrow #gamma#gamma")

# =======================
# Residual Pad (Data-Bkg vs Fit-Bkg)
# =======================
residPad = ROOT.TPad("residPad", "residPad", 0, 0.05, 1, 0.3)
residPad.SetTopMargin(0)
residPad.SetBottomMargin(0.3)
residPad.Draw()
residPad.cd()

# Style
resid.SetMarkerStyle(20)
resid.SetMarkerSize(0.5)
resid.SetLineColor(ROOT.kBlack)
resid.GetYaxis().SetTitle("Events - Bkg")
resid.GetYaxis().SetTitleSize(0.08)
resid.GetYaxis().SetTitleOffset(0.5)
resid.GetYaxis().SetLabelSize(0.08)
resid.GetXaxis().SetTitle("di-photon invariant mass m_{#gamma#gamma} [GeV]")
resid.GetXaxis().SetTitleSize(0.10)
resid.GetXaxis().SetLabelSize(0.10)
resid.SetStats(0)

# Draw residuals (data - bkg)
resid.Draw("E1")

# Zero line
zeroLine = ROOT.TLine(100, 0, 160, 0)
zeroLine.SetLineColor(ROOT.kRed)
zeroLine.SetLineStyle(2)
zeroLine.Draw("same")

gaussian_curve = frame.findObject("gaussian_curve")
gaussian_curve.Draw("same")

canvas.Draw()

# If you want to save a copy of your plot locally as a PDF, uncomment the following line
# You can save the plot as a different file type (png, jpg, eps, etc) by changing the file name
# c.SaveAs("Hyy_fit_analysis.pdf")

And there it is: a clear peak in the invariant mass spectrum around 125 GeV, signaling the presence of the Higgs boson!

This represents a very close result to the one obtained by ATLAS in [Higgs decay to photons measurement paper](https://journals.aps.org/prd/abstract/10.1103/PhysRevD.98.052005), which can be seen below:

<CENTER><img src="https://github.com/atlas-outreach-data-tools/notebooks-collection-opendata/blob/master/13-TeV-examples/uproot_python/images/ImagesHiggs/HyyPaper.png?raw=1" style="width:50%"></CENTER>

Now we can stop our timer and report how long this all took us.

In [ ]:
end = time.time()
duration = end-start
print(f"Finished in {int(duration//60)} min {duration%60} s")

While our main task may be done, there's still more to explore. Here are some additional tasks you can try with this notebook:\n",
* Check how many events are being thrown away by each selection in the analysis;
* Add more selections from the [Higgs boson discovery paper](https://www.sciencedirect.com/science/article/pii/S037026931200857X#se0090) or the [Higgs decay to photons measurement paper](https://journals.aps.org/prd/abstract/10.1103/PhysRevD.98.052005);
* Try different functions for the fit;
* Or anything else you'd like!


<div class="alert alert-block alert-info">
We welcome your feedback on this notebook or any of our other materials! Please <a href="https://forms.gle/zKBqS1opAHHemv9U7">fill out this survey</a> to let us know how we're doing, and you can enter a raffle to win some <a href="https://atlas-secretariat.web.cern.ch/merchandise">ATLAS merchandise</a>
</div>